In [1]:
# 202 - Create SparkSession

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerChurn") \
    .master("local[2]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [2]:
# 203 - Defining Manual Schema

from pyspark.sql.types import *

schema = StructType([
    StructField("customerID", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("SeniorCitizen", IntegerType(), True),
    StructField("Partner", StringType(), True),
    StructField("Dependents", StringType(), True),
    StructField("tenure", IntegerType(), True),
    StructField("PhoneService", StringType(), True),
    StructField("MultipleLines", StringType(), True),
    StructField("InternetService", StringType(), True),
    StructField("OnlineSecurity", StringType(), True),
    StructField("OnlineBackup", StringType(), True),
    StructField("DeviceProtection", StringType(), True),
    StructField("TechSupport", StringType(), True),
    StructField("StreamingTV", StringType(), True),
    StructField("StreamingMovies", StringType(), True),
    StructField("Contract", StringType(), True),
    StructField("PaperlessBilling", StringType(), True),
    StructField("PaymentMethod", StringType(), True),
    StructField("MonthlyCharges", DoubleType(), True),
    StructField("TotalCharges", StringType(), True),
    StructField("Churn", StringType(), True)
])

In [5]:
# 204 - read csv

df = spark.read.csv(
    r"D:\projectpart1\WA_Fn-UseC_-Telco-Customer-Churn.csv",
    header=True,
    schema = schema
)

In [6]:
import pandas as pd
pdf = pd.read_csv(r"D:\projectpart1\WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [8]:
print(df.printSchema())
print(pdf.dtypes)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

None
customerID           object
gender               object
SeniorCitizen       

In [9]:
# 205 - fix totalCharges

from pyspark.sql import functions as F

df = df.withColumn(
    "TotalCharges",
    F.regexp_replace(F.col("TotalCharges"), r"^\s*$", "0")
    .cast("double")
)

In [10]:
# 206 - encode churn

df = df.withColumn(
    "churn_encoded",
    F.when(F.col("Churn") == "Yes", 1).otherwise(0)
)

In [11]:
# 207 - group by contract

contract_summary = df.groupBy("Contract").agg(
    F.count("*").alias("customer_count"),
    F.avg("churn_encoded").alias("churn_rate")
)

contract_summary.show()

+--------------+--------------+-------------------+
|      Contract|customer_count|         churn_rate|
+--------------+--------------+-------------------+
|Month-to-month|          3875| 0.4270967741935484|
|      One year|          1473|0.11269517990495587|
|      Two year|          1695|0.02831858407079646|
+--------------+--------------+-------------------+



In [13]:
# 208 - group by internetService

contract_summary = df.groupBy("internetService").agg(
    F.count("*").alias("customer_count"),
    F.avg("churn_encoded").alias("churn_rate")
)

contract_summary.show()

+---------------+--------------+-------------------+
|internetService|customer_count|         churn_rate|
+---------------+--------------+-------------------+
|    Fiber optic|          3096| 0.4189276485788114|
|             No|          1526|0.07404980340760157|
|            DSL|          2421| 0.1895910780669145|
+---------------+--------------+-------------------+



In [14]:
# 209 - cross validate

pandas_contract = pdf.groupby("Contract")["Churn"].apply(
    lambda x: (x == "Yes").mean()
)

In [ ]:
import os

os.environ["HADOOP_HOME"] = r"D:\practice setup\pyspark-bigdata\hadoop"


In [23]:
print("Hadoop:", os.path.exists(r"D:\practice setup\pyspark-bigdata\hadoop\bin"))
print("winutils:",os.path.exists(r"D:\practice setup\pyspark-bigdata\hadoop\bin\winutils.exe"))

Hadoop: True
winutils: True


In [34]:
# 210 - write parquet
!pip install fastparquet
!pip install pyarrow

import sys

!{sys.executable} -m pip install fastparquet pyarrow


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\aravind.harish\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\aravind.harish\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


  Obtaining dependency information for fastparquet from https://files.pythonhosted.org/packages/08/fe/ba47bfa67868c42f3203e6738b90b80e25f3ecdf1f85e253b27ef2135cf2/fastparquet-2026.5.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for pyarrow from https://files.pythonhosted.org/packages/65/ee/197cbf47e49f83e6ebeb946a5259a48a638dea27ac774db42fe78022179d/pyarrow-25.0.1-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for cramjam>=2.3 from https://files.pythonhosted.org/packages/75/b1/d53002729cfd94c5844ddfaf1233c86d29f2dbfc1b764a6562c41c044199/cramjam-2.11.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for fsspec from https://files.pythonhosted.org/packages/fd/3c/6a2bf344106328fd04963664a60b9bb6496fc25df8e962fcdc1367285fb9/fsspec-2026.7.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/695.7 kB ? eta -:--:--
   - -------------------------------------- 30.7/695.7 kB 1.3 MB/s eta 0:00:01
   -----


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
output_dir = r"D:\projectpart1\telco2\Project\spark_output\summary_con"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "data.parquet")

contract_summary.toPandas().to_parquet(
    file_path,
    engine="fastparquet",
    index=False)

In [36]:
# 211 - read parquet and validate round-trip

read_back = spark.read.parquet(file_path)

read_back.printSchema()

print("Row count:", read_back.count())

root
 |-- internetService: string (nullable = true)
 |-- customer_count: long (nullable = true)
 |-- churn_rate: double (nullable = true)

Row count: 3
